<div style="text-align: center;">
    <h1><strong>Detección de <i>Clickbait</i> en noticias</strong></h1>
    <h2>Proyecto Procesado de Lenguaje Natural</h2>
    <br>
    <h3>Máster en Ciencia de Datos</h3>
    <h4>Universitat de València</h4>
    <br>
    <p>Juan Alcaráz Otón, Xueyao An, Fabián Calvo Castillo, Adrián Carrasco Alcalá, Javier Herrero Pérez, Mario Martínez Guillén y Clara Montalvá Barcenilla</p>
    <p><b>Curso 2025/2026</p>
</div>
<hr>

## Introducción

El periodismo digital ha transformado profundamente la forma en que los usuarios consumen información, generando un ecosistema altamente competitivo donde la atención del lector es el principal activo. 
En este contexto, ha proliferado el uso del clickbait, que esto se define como una estrategia utilizada en los medios digitales que busca llamar la atención a través de los titulares, apelando a las emociones y a la curiosidad de los lectores para forzar el click en la noticia, a menudo en perjuicio de la calidad informativa (Bravo Araujo, Serrano-Puche y Novoa Jaso, 2021).

De acuerdo con Bazaco et al. (2019), se trata de un fenómeno comunicativo dinámico que prioriza la interrogante sobre la información, omitiendo datos clave para generar un vacío de curiosidad o empleando un enfoque sensacionalista. Dentro del marco de este proyecto, se adoptará una aproximación dual para la identificación de clickbait:

* Por un lado, como un titular sensacionalista diseñado con estructuras lingüísticas concretas para captar la atención e incentivar el click, el cual puede ser detectado analizando únicamente el titular.

* Por otro lado, como un titular llamativo que presenta una diferencia considerable con el contenido de la noticia, ya sea porque plantea una pregunta que nunca se responde o porque presenta información completamnete opuesta a la que se desarrolla en el cuerpo del texto.

Para automatizar la detección de estos patrones, el proyecto emplea el modelo taniwasl/clickbait_es, alojado en Hugging Face. Este clasificador se fundamenta en BETO, la versión entrenada para el idioma español del modelo de lenguaje BERT (Bidirectional Encoder Representations from Transformers). Este modelo ha sido ajustado (fine-tuned) específicamente con un corpus aproximado de 30.000 noticias provenientes de múltiples medios españoles. 

Su funcionamiento radica en procesar la secuencia de palabras del titular analizando el contexto bidireccional de cada término mediante mecanismos de atención. Al haber aprendido de decenas de miles de ejemplos, el modelo es capaz de ponderar características semánticas y sintácticas intrínsecas del clickbait, permitiendo predecir y clasificar nuevos titulares con precisión.


## Objetivo

Objetivo General:

* Analizar y clasificar el uso de técnicas de clickbait en la prensa digital española mediante la extracción automatizada de noticias y la aplicación de técnicas de Procesamiento de Lenguaje Natural.

Objetivos Específicos:

1. Implementar técnicas de web scraping para adquirir de forma automatizada noticias de las categorías Internacional, Nacional y Cultura procedentes de una selección de los principales periódicos y medios de comunicación españoles.

2. Analizar los titulares y los contenidos textuales de las noticias desde la perspectiva del NLP para extraer características distintivas que diferencien las noticias con y sin clickbait, evaluando las tendencias en función del medio y la categoría.

3. Integrar un agente de Inteligencia Artificial que automatice la inferencia y el etiquetado masivo de los titulares adquiridos, clasificándolos en clickbait o no clickbait y con la capacidad añadida de generar un titular que adecuado a las noticias clasificadas como clickbait.


## Metodología

### 1. Adquisición de los Datos

Obtenemos noticias de los siguientes periódicos y medios españoles:

- ABC
- elDiario
- El Confidencial
- La Vanguardia
- 20minutos
- OkDiario
- RTVE
- Mediterráneo Digital
- El HuffPost

En particular, los periódicos digitales _Mediterráneo Digital_ y _El Huffpost_ destacan por su gran cantidad de titulares sensacionalistas y que no se corresponden con el contenido de los cuerpos de las noticias.

Extraemos los artículos de las siguientes categorías:

- Internacional: Noticias de ámbito mundial
- Nacional: Noticias de España y de sus regiones
- Cultura: Artes, cine, literatura, entretenimiento, etc.

Para ello, utilizamos tanto las páginas feed de aquellos periódicos que disponen de ellas como técnicas de scraping directo sobre los HTML de las páginas web.

Las noticias extraídas se almacenan en formato JSON con la siguiente estructura:

```json
{
  "Link": "string",
  "Periódico": "string",
  "Fecha": "string (YYYY-MM-DD)",
  "Título": "string",
  "Subtítulo": "string o null",
  "Categoría": "string",
  "Contenido": "string"
}
```

Los campos de esta estructura representan lo siguiente:

| Campo | Tipo | Descripción |
|-------|------|-------------|
| Link | string | URL del artículo original |
| Periódico | string | Nombre del medio de comunicación |
| Fecha | string | Fecha de publicación (formato YYYY-MM-DD) |
| Título | string | Título principal del artículo |
| Subtítulo | string o null | Subtítulo o descripción breve |
| Categoría | string | Categoría o sección del artículo |
| Contenido | string | Texto completo del artículo |

Se crea un JSON para cada periódico, en el que se recogen todas sus noticias, y estos ficheros se guardan en la carpeta 'data/' con el formato de nombre 'nombredelmedio.json'.

### 2. Preprocesado del Texto

Una vez obtenidas suficientes noticias de los diferentes medios, las guardamos todas en el archivo JSON común 'conjunto_noticias.json'.

Tras una vista previa del contenido almacenado en dicho archivo, se determina que los elementos del texto que deberían eliminarse son los siguientes:

- **Comillas:** hay muchas formas diferentes que cada periódico utiliza para poner comillas, estas son: /""/, ««, '', "" y algunas más.  
- **Signos de interrogación y exclamación:** ¡! ¿?
- **Guiones:** --
- **URLs**
- **Direcciones de correo y cuentas de twitter:** empiezan por @
- **Saltos de línea:** \n \r
- **Signos de puntuación:** , ; : .
- **Corchetes y paréntesis:** [] ()
- **Emojis**

### 3. Clasificación de las Noticias en Clickbait/No Clickbait

Una vez finalizado el preprocesado del texto y consolidado el conjunto de datos limpio en conjunto_noticias.json, se procede a la etapa de clasificación automática de los titulares. Para esta tarea central, se implementa el modelo preentrenado taniwasl/clickbait_es, disponible en el repositorio de Hugging Face.

El procedimiento técnico de clasificación consta de los siguientes pasos:

1. Carga del entorno de inferencia: A través de la librería transformers, se instancia tanto el tokenizador correspondiente a la arquitectura BETO como el propio modelo de clasificación de secuencias (AutoModelForSequenceClassification).

2. Tokenización de titulares: Cada valor del campo "Título" de nuestro JSON se pasa por el tokenizador. Este paso convierte el texto crudo en tensores y añade los tokens especiales de inicio y fin de oración que requiere la red neuronal para acotar el contexto.

3. Inferencia del modelo: Los tensores se introducen en el modelo preentrenado. Gracias a su fine-tuning previo sobre noticias de la prensa española, el modelo hace pasar las representaciones vectoriales por sus distintas capas ocultas. El sistema detecta patrones típicos del sensacionalismo en español como la hiperbolización, deícticos temporales o preguntas abiertas.

4. Etiquerado: La red neuronal devuelve unos valores que determinan la probabilidad de que el texto pertenezca a la clase clickbait o a la clase no clickbait. A cada noticia en el conjunto de datos se le adjunta esta nueva etiqueta predictiva.

La automatización de este proceso nos permite generar la variable objetivo para todo el conjunto de noticias. Con los datos ya etiquetados, es posible realizar un análisis cruzado para validar estadísticamente qué medios y categorías temáticas incurren con mayor frecuencia en la desinformación o en tácticas de clickbait.


### 4. Extracción de características

#### 4.1. Enfoque inicial: características lingüisticas


Se inició el análisis calculando la distancia coseno entre los embeddings del título y los fragmentos del cuerpo de cada noticia. Con este enfoque se buscaba capturar la relación semántica entre el titular y el contenido, bajo la hipótesis de que los artículos con clickbait presentarían mayores discrepancias semánticas entre ambos elementos.

Se exploraron variables lingüísticas más sofisticadas, incluyendo deícticos, intensificadores,superlativos, palabras emocionales y frases de curiosidad. Estas características fueron
seleccionadas por su relevancia teórica en la detección de clickbait, al ser indicadores habituales en titulares sensacionalistas.

#### 4.2. Exploración de características alternativas


Se evaluaron diferentes enfoques con el objetivo de mejorar la capacidad discriminativa del modelo: Análisis de entropía, se calculó la entropía como medida de redundancia léxica, buscando patrones en la repetición de palabras entre títulos y contenido; Análisis de sentimientos, se aplicaron métodos de análisis de sentimiento basados en BERT, bajo la hipótesis de que el clickbait podría presentar patrones emocionales diferenciados; y otras métricas, se exploraron índices de legibilidad (Flesch-Kincaid), longitud de textos y métodos de pregunta-respuesta.

Ninguna de estas características logró producir diferencias estadísticas significativas entre los grupos, lo que motivó la transición hacia un enfoque basado en representaciones neuronales.

### 5. Fine Tuning con BERT


Ante los resultados limitados del enfoque basado en características manuales, se recurrió al fine-tuning de un modelo de transformers. Se utilizó **bert-base-multilingual-cased** como modelo base, optimizado para el procesamiento de texto en múltiples idiomas, incluyendo el español.

La configuración del entrenamiento fue la siguiente: Número de épocas 3-4 según el modelo; learning rate de $2 \cdot 10^{-5}$, optimizador Adam, división de datos $80 \%$ entrenamiento y $20 \%$ validación; tamaño de batch de 16 para títulos y 8 para contenido; corrección de desbalance mediante pesos de clase inversamente proporcionales a la frecuencia; y un dropout de 0.2 para evitar sobreajuste.

Se entrenano el modelo durante tan pocas épocas para evitar modificar demasiado los pesos del modelo preentrenado, dado el tamaño limitado del corpus. El objetivo es que el modelo aprenda patrones específicos de clickbait sin perder la estructura semántica general de las representaciones de BERT.

Se realizó el fine-tuning de forma independiente para títulos y contenido, con longitudes máximas de secuencia de 128 y 512 tokens respectivamente. Los embeddings de clasificación fueron extraídos del token *[CLS]*.

### 6. Validación estadística

Para validar la significancia estadística de la separación observada, se aplicó la **prueba U de Mann-Whitney**, un test no paramétrico que contrasta si dos distribuciones independientes son estadísticamente diferentes. Se calculó el p-valor para ambas dimensiones UMAP de forma independiente, aplicando corrección de Bonferroni para comparaciones múltiples. 

### 6. Implementación Agéntica

## Resultados

## Extracción de características

A partir de la distancia coseno entre el título y los párrafos del contenido, combinada con las diferencias lingüísticas descritas anteriormente, se entrenó un clasificador mediante regresión logística. Otros clasificadores más complejos mostraron sobreajuste, por lo que se optó por este modelo más simple.

<div style="display:flex; gap:20px;">

<div style="flex:1; text-align:center;">

<img src="img/embeding_coseno_simple.png" width="75%">

<p><strong>Figura 1:</strong> Espacio embebido UMAP que relaciona el título con el cuerpo de la noticia. El número encima de cada punto corresponde al párrafo del cuerpo.</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/ROC_caracteristicas.png" width="70%">

<p><strong>Figura 2:</strong> Curva ROC del clasificador basado en características lingüísticas (AUC = 0.660).</p>

</div>

</div>



En la figura 1 se observa que las noticias con mayor similitud coseno entre título y contenido tienden a agruparse en regiones próximas del espacio embebido. Sin embargo, como refleja la figura 2, la capacidad discriminativa del clasificador resultante es limitada (AUC = 0.660), lo que evidencia que las características lingüísticas manuales no son suficientes para una detección robusta del clickbait.

## Embedding tras fine-tuning con etiquetas Taniwa

Tras el fine-tuning con las etiquetas del sistema Taniwa, los embeddings resultantes
muestran una separación clara entre ambas clases. La Figura \ref{fig:umap_finetuned_taniwa}
revela clusters bien definidos de clickbait y no-clickbait, tanto para títulos como para
contenido.

<div style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_clickbait_taniwa.png" width="90%">

<p>
<strong>Figura 1.</strong>
Embeddings UMAP tras fine-tuning para títulos (Taniwa).
Se aprecia separación clara entre clusters.
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_clickbait_taniwa.png" width="90%">

<p>
<strong>Figura 2.</strong>
Embeddings UMAP tras fine-tuning para contenido (Taniwa).
</p>

</div>

</div>

<p align="center">
<strong>Figura 3.</strong>
Representación UMAP de embeddings tras el fine-tuning supervisado con etiquetas Taniwa.
</p>

La separación visual observada fue validada mediante la prueba U de Mann-Whitney,
que confirmó significancia estadística ($p \ll 0.05$) en ambas dimensiones UMAP.
Cabe recordar que esta separación es en parte consecuencia del fine-tuning supervisado:
el modelo optimiza sus representaciones para discriminar las etiquetas de entrenamiento,
por lo que la separación en el espacio embebido no implica necesariamente que el clickbait
sea intrínsecamente distinguible a nivel lingüístico.

## Variación por categoría temática (Taniwa)

La Figura [3](#fig-umap-finetuned-categoria-taniwa) presenta los espacios embebidos
segmentados por categoría temática, usando las etiquetas de Taniwa.

<div id="fig-umap-finetuned-categoria-taniwa" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_categoria_taniwa.png" width="100%">

<p>
<strong>Figura 3.</strong>
Títulos por categoría temática (Taniwa).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_categoria_taniwa.png" width="100%">

<p>
<strong>Figura 4.</strong>
Contenido por categoría temática (Taniwa).
</p>

</div>

</div>

<p align="center">
<strong>Figura 5.</strong>
Representación UMAP segmentada por categoría temática usando etiquetas Taniwa.
</p>

La categoría *Cultura* presenta una mayor propensión hacia el clickbait, evidenciada
por su concentración en la región del espacio UMAP asociada a esta etiqueta. Las categorías
*Internacional* y *Nacional* muestran una distribución más equilibrada entre
ambas clases.


## Variación por medio de comunicación (Taniwa)

La Figura [6](#fig-umap-finetuned-periodico-taniwa) presenta la distribución segmentada por
medio de comunicación. Los resultados no evidencian que ningún periódico concentre sus
contenidos de forma sistemática en una sola clase.

<div id="fig-umap-finetuned-periodico-taniwa" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_periodico_taniwa.png" width="100%">

<p>
<strong>Figura 6.</strong>
Títulos por medio de comunicación (Taniwa).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_periodico_taniwa.png" width="100%">

<p>
<strong>Figura 7.</strong>
Contenido por medio de comunicación (Taniwa).
</p>

</div>

</div>

<p align="center">
<strong>Figura 8.</strong>
Representación UMAP segmentada por medio de comunicación usando etiquetas Taniwa.
</p>

Los periódicos analizados presentan distribuciones heterogéneas en ambas regiones del espacio,
sin que exista un medio que muestre una preferencia marcada hacia el clickbait.

## Embeddings tras fine-tuning con etiquetas del agente GPT

Se repitió el análisis utilizando las etiquetas generadas por el agente GPT, que clasifica
las noticias teniendo en cuenta tanto el título como el contenido. La
Figura [9](#fig-umap-finetuned-gpt) muestra los resultados obtenidos.

<div id="fig-umap-finetuned-gpt" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_clickbait_gpt.png" width="100%">

<p>
<strong>Figura 9.</strong>
Embeddings UMAP tras fine-tuning para títulos (GPT).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_clickbait_gpt.png" width="100%">

<p>
<strong>Figura 10.</strong>
Embeddings UMAP tras fine-tuning para contenido (GPT).
</p>

</div>

</div>

<p align="center">
<strong>Figura 11.</strong>
Representación UMAP de embeddings tras el fine-tuning supervisado con etiquetas GPT.
</p>

Al igual que con las etiquetas de Taniwa, se observa una separación clara entre ambas
clases en el espacio UMAP, validada estadísticamente con la prueba U de Mann-Whitney
($p \ll 0.05$).



## Variación por categoría temática (GPT)

La Figura [12](#fig-umap-finetuned-categoria-gpt) muestra la distribución por categoría
temática con las etiquetas del agente GPT. Los patrones son coherentes con los observados
en el análisis con etiquetas Taniwa: la categoría *Cultura* concentra más contenidos
clasificados como clickbait, mientras que *Internacional* y *Nacional* presentan
distribuciones más equilibradas.

<div id="fig-umap-finetuned-categoria-gpt" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_categoria_gpt.png" width="100%">

<p>
<strong>Figura 12.</strong>
Títulos por categoría temática (GPT).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_categoria_gpt.png" width="100%">

<p>
<strong>Figura 13.</strong>
Contenido por categoría temática (GPT).
</p>

</div>

</div>

<p align="center">
<strong>Figura 14.</strong>
Representación UMAP segmentada por categoría temática usando etiquetas GPT.
</p>


## Variación por medio de comunicación (GPT)

La Figura [15](#fig-umap-finetuned-periodico-gpt) presenta la distribución por medio de
comunicación con etiquetas GPT. De forma consistente con el análisis anterior, no se
observa ningún periódico con una preferencia sistemática hacia el clickbait.

<div id="fig-umap-finetuned-periodico-gpt" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_periodico_gpt.png" width="100%">

<p>
<strong>Figura 15.</strong>
Títulos por medio de comunicación (GPT).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_periodico_gpt.png" width="100%">

<p>
<strong>Figura 16.</strong>
Contenido por medio de comunicación (GPT).
</p>

</div>

</div>

<p align="center">
<strong>Figura 17.</strong>
Representación UMAP segmentada por medio de comunicación usando etiquetas GPT.
</p>


## Comparación entre títulos y contenido

En ambos sistemas de etiquetado (Taniwa y GPT), el modelo entrenado sobre títulos mostró
una convergencia más estable durante el entrenamiento y una discriminación más robusta
en la comparación posterior. Este resultado es coherente con la naturaleza del clickbait
como fenómeno del titular [Chakraborty et al., 2016]: el título es el elemento diseñado
para generar curiosidad o engañar, mientras que el cuerpo de la noticia tiende a ser
más informativo y neutral.

El modelo de contenido presenta mayor inestabilidad en las curvas de entrenamiento, lo
que puede atribuirse a la longitud de los textos (hasta 512 tokens), al tamaño reducido
del corpus y al truncado que descarta información de las noticias más largas.


## Comparación entre clasificadores: BERT vs. Taniwa

Para evaluar en qué medida el modelo BERT (entrenado con etiquetas GPT) reproduce las
decisiones del sistema Taniwa, se realizó una inferencia sobre el corpus de Taniwa y se
compararon las clasificaciones mediante el coeficiente Kappa de Cohen [Cohen, 1960].



<div id="fig-umap-finetuned-gpt" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/agente_acuerdo_titulo.png" width="100%">

<p>
<strong>Figura 9.</strong>
Embeddings UMAP tras fine-tuning para títulos (GPT).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/agente_acuerdo_contenido.png" width="100%">

<p>
<strong>Figura 10.</strong>
Embeddings UMAP tras fine-tuning para contenido (GPT).
</p>

</div>

</div>

<p align="center">
<strong>Figura 11.</strong>
Representación UMAP de embeddings tras el fine-tuning supervisado con etiquetas GPT.
</p>

Los resultados se resumen en la Tabla 1.

<div id="tab-kappa" style="display:flex; justify-content:center; margin:20px 0;">

<table>
<thead>
<tr>
<th>Métrica</th>
<th>Títulos</th>
<th>Contenido</th>
</tr>
</thead>

<tbody>
<tr>
<td><strong>Acuerdo simple</strong></td>
<td>82.0%</td>
<td>62.6%</td>
</tr>

<tr>
<td><strong>Cohen's Kappa</strong></td>
<td>0.405</td>
<td>0.023</td>
</tr>

<tr>
<td><strong>Interpretación</strong></td>
<td>Moderado</td>
<td>Bajo</td>
</tr>
</tbody>
</table>

</div>

<p align="center">
<strong>Tabla 1.</strong>
Acuerdo entre BERT (entrenado con GPT) y Taniwa.
</p>

Un Kappa de aproximadamente $0.41$ indica un acuerdo moderado [Landis y Koch, 1977],
lo que refleja que ambos clasificadores comparten el criterio de detección de clickbait
de forma parcial pero no completa. Esta divergencia es esperable dado que se trata de
aproximaciones distintas: GPT razona de forma holística sobre el texto completo, mientras
que BERT aprende patrones estadísticos locales a partir de un corpus limitado.

El mayor desacuerdo se concentra en el *HuffPost* y en la categoría
*Cultura*, lo que sugiere que el clickbait en estos contextos presenta
características más ambiguas o estilísticas que los modelos interpretan de forma
diferente.

## Conclusiones

## Bibliografía

1. Bazaco, A., Redondo, M., y Sánchez-García, P. (2019). El clickbait como estrategia del periodismo viral: concepto y metodología. Revista Latina de Comunicación Social, (74), 94-115.

2. Bravo Araujo, A., Serrano-Puche, J., y Novoa Jaso, M. (2021). Uso del clickbait en los medios nativos digitales españoles. Un análisis de El Confidencial, El Español, Eldiario.es y Ok Diario. Doxa Comunicación. Revista Interdisciplinar de Estudios de Comunicación y Ciencias Sociales, (32), 185-210.

3. Taniwa (2023). taniwasl/clickbait_es. Hugging Face. Recuperado de https://huggingface.co/taniwasl/clickbait_es

4. Devlin, J., Chang, M.-W., Lee, K., y Toutanova, K. (2018).  
   *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*.  
   CoRR, abs/1810.04805.  
   Disponible en: http://arxiv.org/abs/1810.04805

5. Mann, H. B., y Whitney, D. R. (1947).  
   *On a Test of Whether one of Two Random Variables is Stochastically Larger than the Other*.  
   The Annals of Mathematical Statistics, 18(1), 50–60.  
   https://doi.org/10.1214/aoms/1177730491


- *Información sobre los periódicos analizados*

COSAS QUE NOS FALTAN METER:

- robots.txt, explciar que lo hemos mirado todos y que no había problema